In [ ]:
!pip install gymnasium[atari,accept-rom-license] stable-baselines3 shimmy
!pip install opencv-python

In [ ]:
import os
import gymnasium as gym
import ale_py
import sys
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.callbacks import CheckpointCallback

# 1. SETUP ENVIRONMENT
def make_env():
    env = gym.make("ALE/Freeway-v5", render_mode="rgb_array", full_action_space=False)
    env = AtariWrapper(env)
    return env

env = DummyVecEnv([make_env])
env = VecFrameStack(env, n_stack=4)

# 2. MODEL 2 CONFIGURATION
model_2 = DQN(
    "CnnPolicy", 
    env, 
    verbose=0,               # Set to 0 so we can control the output manually
    buffer_size=150000, 
    learning_rate=1e-4, 
    batch_size=64, 
    gamma=0.99,
    target_update_interval=2000,
    policy_kwargs={'net_arch': [256, 256]}, 
    device="auto"
)

# 3. DIRECTORY & CALLBACK
save_path = "./freeway_model_2_advanced/"
if not os.path.exists(save_path): os.makedirs(save_path)
checkpoint_callback = CheckpointCallback(save_freq=25000, save_path=save_path, name_prefix="model2_500k")

# 4. TRAINING WITH REFRESHING OUTPUT
print("🚀 Training Model 2 (Dueling + Frame Stack)...")
print("Progress: [--------------------] 0%", end="")

try:
    # Stable Baselines 3's built-in progress_bar=True actually refreshes the line automatically!
    # But if you want a custom refreshing status, we use the .learn() method like this:
    model_2.learn(
        total_timesteps=200000, 
        callback=checkpoint_callback,
        progress_bar=True  # This provides a single-line refreshing bar by default
    )
    
    model_2.save(os.path.join(save_path, "model_2_final"))
    print(f"\n\n✅ Training Complete! Model saved to {save_path}")

except Exception as e:
    print(f"\n❌ Error during training: {e}")

In [ ]:
import os
import gymnasium as gym
import ale_py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

# 1. SETUP ENVIRONMENT (Must match Model 2's training: FrameStack=4)
def make_env():
    env = gym.make("ALE/Freeway-v5", render_mode="rgb_array", full_action_space=False)
    env = AtariWrapper(env)
    return env

env = DummyVecEnv([make_env])
env = VecFrameStack(env, n_stack=4)

# 2. LOAD MODEL 2
model_2_path = "./freeway_model_2_advanced/model_2_final"
model = DQN.load(model_2_path)

num_episodes = 30
m2_data = []

print(f"📊 Evaluating Model 2 (The Challenger)...")
print(f"{'Episode':<10} | {'Crossings':<10} | {'Sec/Cross':<10}")
print("-" * 40)

for i in range(num_episodes):
    obs = env.reset()
    done = [False]
    score = 0
    start_time = time.time()
    
    while not done[0]:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = env.step(action)
        score += reward[0]
    
    duration = time.time() - start_time
    time_per_cross = duration / score if score > 0 else 126.0
    
    print(f"{i+1:<10} | {score:<10} | {time_per_cross:<10.2f}")
    m2_data.append({'Score': score, 'Speed': time_per_cross})

df2 = pd.DataFrame(m2_data)

# 3. STATISTICAL CALCULATIONS
# (Replace these with your actual Model 1 results from earlier)
m1_mean = 21.0  # Example placeholder
m1_ci = 1.5     # Example placeholder

m2_mean = df2['Score'].mean()
m2_std = df2['Score'].std()
m2_ci = 1.96 * (m2_std / np.sqrt(num_episodes))

# 4. THE COMPARISON GRAPH (Level 4 Requirement)
labels = ['Model 1 (Baseline)', 'Model 2 (Dueling + Stack)']
means = [m1_mean, m2_mean]
errors = [m1_ci, m2_ci]

plt.figure(figsize=(8, 6))
bars = plt.bar(labels, means, yerr=errors, capsize=10, color=['lightgray', 'gold'], edgecolor='black')
plt.ylabel('Mean Crossings (Score)')
plt.title('Final Performance Comparison: Model 1 vs Model 2')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, f'{yval:.2f}', ha='center', fontweight='bold')

plt.show()

print("\n" + "="*30)
print("HYPOTHESIS TEST DATA")
print("="*30)
print(f"Model 2 Mean: {m2_mean:.2f}")
print(f"Model 2 95% CI: [{m2_mean - m2_ci:.2f}, {m2_mean + m2_ci:.2f}]")
if (m2_mean - m2_ci) > (m1_mean + m1_ci):
    print("✅ RESULT: Statistically Significant Improvement (No CI Overlap)")
else:
    print("⚠️ RESULT: Improvement not statistically significant at 95% level.")
print("="*30)

In [ ]:
import gymnasium as gym
import ale_py
import time
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

# 1. SETUP THE "LIVE" ENVIRONMENT
def make_env():
    # We use 'human' render_mode to actually see the game
    env = gym.make("ALE/Freeway-v5", render_mode="human", full_action_space=False)
    env = AtariWrapper(env)
    return env

# Must stack 4 frames because Model 2 was trained to see motion!
env = DummyVecEnv([make_env])
env = VecFrameStack(env, n_stack=4)

# 2. LOAD YOUR BEST AGENT
model = DQN.load("./freeway_model_2_advanced/model_2_final")

# 3. RUN THE DEMO
print("🎮 Starting Live Speedrun Demo...")
obs = env.reset()
done = [False]
total_crossings = 0

try:
    while not done[0]:
        # Deterministic=True ensures the agent uses its best learned path
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = env.step(action)
        
        if reward[0] > 0:
            total_crossings += 1
            print(f"✅ Crossing Successful! Total: {total_crossings}")
        
        # Slow down slightly so humans can watch the magic happen
        time.sleep(0.01) 

except KeyboardInterrupt:
    print("\nDemo stopped by user.")
finally:
    env.close()
    print(f"🏁 Final Score in Demo: {total_crossings}")